In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

# 경로 설정
zip_file_path = '/content/drive/MyDrive/aihub_final_dataset.zip'
extract_root = '/content/dataset'

# 로컬로 압축 해제
if not os.path.exists(extract_root):
    os.makedirs(extract_root)
    print("로컬(SSD)로 압축 해제 시작 (학습 속도 향상)...")
    !unzip -q "{zip_file_path}" -d "{extract_root}"
    print("압축 해제 완료!")
else:
    print("이미 로컬에 압축이 풀려 있습니다.")

dataset_path = extract_root
print(f"학습 데이터셋 기준 경로: {dataset_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 로컬(SSD)로 압축 해제 시작 (학습 속도 향상)...
✅ 압축 해제 완료!
학습 데이터셋 기준 경로: /content/dataset


In [ ]:
import os
import glob

# 하위 폴더까지 모두 검색하기 위해 recursive=True와 ** 사용
train_labels = glob.glob('/content/dataset/aihub_dataset/labels/train/**/*.txt', recursive=True)
valid_labels = glob.glob('/content/dataset/aihub_dataset/labels/valid/**/*.txt', recursive=True)
all_labels = train_labels + valid_labels

class_counts = {0: 0, 1: 0}
total_boxes = 0

print(f"총 {len(all_labels)}개의 라벨 파일 분석 시작...")

for label_path in all_labels:
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue

                class_id = int(parts[0])
                if class_id in class_counts:
                    class_counts[class_id] += 1
                    total_boxes += 1
    except Exception as e:
        print(f"Error reading {label_path}: {e}")

print("=" * 50)
print(f"[라벨 전수조사 결과] 총 라벨 파일 수: {len(all_labels)}개")
print(f"전체 Bounding Box 개수: {total_boxes}개")
print("-" * 50)
print(f"Class 0 (YES_Helmet) 개수 : {class_counts[0]:,}개")
print(f"Class 1 (NO_Helmet) 개수   : {class_counts[1]:,}개")
print("=" * 50)

if total_boxes > 0:
    ratio0 = (class_counts[0] / total_boxes) * 100
    ratio1 = (class_counts[1] / total_boxes) * 100
    print(f"분포 비율 -> YES: {ratio0:.1f}%, NO: {ratio1:.1f}%")

🚀 총 50534개의 라벨 파일 분석 시작...
📈 [라벨 전수조사 결과] 총 라벨 파일 수: 50534개
📦 전체 Bounding Box 개수: 82055개
--------------------------------------------------
🪖 Class 0 (YES_Helmet) 개수 : 80,346개
🧑 Class 1 (NO_Helmet) 개수   : 1,709개
분포 비율 -> YES: 97.9%, NO: 2.1%


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.0 MB/s eta 0:00:00


In [ ]:
import yaml

# 데이터셋이 있는 경로
dataset_root = '/content/dataset'

# YAML 데이터 구성
data_config = {
    'path': dataset_root,        # 데이터셋 루트 경로
    'train': 'aihub_dataset/images/train',   # train 이미지 경로
    'val': 'aihub_dataset/images/valid',     # valid 이미지 경로

    'names': {
        0: 'YES_Helmet',
        1: 'NO_Helmet'
    }
}

# 3. 파일 저장
yaml_path = os.path.join(dataset_root, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"data.yaml 파일 생성 완료: {yaml_path}")

✅ data.yaml 파일 생성 완료: /content/dataset/data.yaml


In [ ]:
from ultralytics import YOLO

# 1. 모델 로드
model = YOLO('yolo26n.pt')

# 2. MuSGD 옵티마이저 기반 최적화 학습
model.train(
    data='/content/dataset/data.yaml',
    epochs=150,
    imgsz=640,
    batch=128,
    device=0,

    # [손실 가중치: MuSGD의 빠른 수렴 속도에 맞춰 조정]
    box=12.0,             # 위치 정밀도 향상
    cls=2.0,              # 분류 정확도
    cls_pw=0.7,           # 불균형 데이터셋 대응 (희귀 클래스 중요도 상승)
    dfl=1.2,              # 분포 정밀도

    # [데이터 증강]
    mosaic=1.0,
    mixup=0.25,           # MuSGD는 데이터가 많을 때 mixup과 궁합이 좋음
    degrees=15.0,
    perspective=0.0005,

    # [옵티마이저 및 학습 관리]
    optimizer='MuSGD',    # 사용자 요청 반영
    momentum=0.9,         # MuSGD/SGD 계열에서 관성 조정
    weight_decay=0.0005,
    patience=40,
    save_period=5,
    project='/content/drive/MyDrive/yolo26n_results',
    name='helmet_yolo26n_musgd'
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=12.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.7, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.2, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.25, mode=train, model=yolo26n.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=helmet_yolo26n_musgd-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=MuSGD, over

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('/content/drive/MyDrive/yolo26n_results/helmet_yolo26n_musgd-2/weights/epoch70.pt')

# 영상 경로 설정
video_path = '/content/Take Time to Take Care (Vehicular Safety).mp4'

# 추론 실행 및 결과 저장
# conf=0.3: 신뢰도 임계값, 원하시는 수치로 조정 가능합니다.
# save=True: 결과를 영상으로 저장합니다.
results = model.predict(
    source=video_path,
    conf=0.3,
    save=True,
    imgsz=640,
    device=0,           # GPU 사용
    stream=True         # 대용량 영상 처리를 위한 스트리밍 방식
)

# 4. 결과 출력
# stream=True를 사용하면 generator 객체가 반환되므로 루프를 돌려야 합니다.
for r in results:
    # 각 프레임별로 추론 결과를 처리하거나 시각화할 수 있습니다.
    # r.plot()을 사용하면 바운딩 박스가 그려진 이미지가 생성됩니다.
    annotated_frame = r.plot()

    # 화면 표시를 원하시면 cv2.imshow를 쓰지만, 코랩에서는
    # 자동으로 save=True 설정에 의해 파일로 저장됩니다.
    pass

print("영상 추론 및 결과 저장 완료!")


video 1/1 (frame 1/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 13.3ms
video 1/1 (frame 2/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.5ms
video 1/1 (frame 3/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.5ms
video 1/1 (frame 4/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.2ms
video 1/1 (frame 5/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.5ms
video 1/1 (frame 6/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.3ms
video 1/1 (frame 7/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.2ms
video 1/1 (frame 8/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.4ms
video 1/1 (frame 9/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.7ms
video 1/1